In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df_adni = pd.read_csv('ADNIMERGE.csv', low_memory=False)
df_adni['VISCODE'] = df_adni['VISCODE'].str.replace('m', '', regex=True).astype(int)
df_dataset = pd.read_excel('dataset_final.xlsx')
df_dataset['VISCODE'] = df_dataset['VISCODE'].str.replace('m', '', regex=True).astype(int)


In [2]:
imputer_stat = []
columns = ['Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV',
           'FDG', 'AV45',
           'CDRSB', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting','RAVLT_perc_forgetting','FAQ', 'MOCA', 'LDELTOTAL', 'DIGITSCOR','TRABSCOR',
           'ABETA', 'PTAU', 'TAU']

for column in columns: 
    df_missing = df_dataset[df_dataset[column].isna()][["RID", "VISCODE", column]]
    # print(df_missing)
    unique_list = df_missing['RID'].unique()
    total_missing = len(df_missing)
    counter =0
    for unique_RID in unique_list:
        df_adni_unique = df_adni[df_adni['RID']== unique_RID][["RID", "VISCODE", column]].dropna()
        df_adni_unique[column] = df_adni_unique[column].astype(str)
        df_adni_unique[column] = df_adni_unique[column].str.replace('[<>]', '', regex=True).astype(float)
        if len(df_adni_unique) >= 3:
            model = LinearRegression()
            model.fit(df_adni_unique[['VISCODE']], df_adni_unique[column] )
            
            df_missing_subset = df_missing.loc[df_missing['RID'] == unique_RID].copy()
            counter += len(df_missing_subset)
            X_missing = df_missing_subset[['VISCODE']]  # VISCODE values for prediction

            df_missing_subset[column] = model.predict(X_missing)

            # Update the original df_missing with the predicted values
            df_dataset.update(df_missing_subset)
    imputer_stat.append({column: {"total_missing": total_missing, "imputed":counter}})


In [ ]:
#new
imputer_stat

[{'Ventricles': {'total_missing': 1326, 'imputed': 1008}},
 {'Hippocampus': {'total_missing': 1766, 'imputed': 1134}},
 {'WholeBrain': {'total_missing': 1188, 'imputed': 947}},
 {'Entorhinal': {'total_missing': 1971, 'imputed': 1133}},
 {'Fusiform': {'total_missing': 1971, 'imputed': 1133}},
 {'MidTemp': {'total_missing': 1971, 'imputed': 1133}},
 {'ICV': {'total_missing': 1048, 'imputed': 879}},
 {'FDG': {'total_missing': 3169, 'imputed': 488}},
 {'AV45': {'total_missing': 4005, 'imputed': 781}},
 {'CDRSB': {'total_missing': 46, 'imputed': 40}},
 {'MMSE': {'total_missing': 29, 'imputed': 25}},
 {'RAVLT_immediate': {'total_missing': 120, 'imputed': 98}},
 {'RAVLT_learning': {'total_missing': 120, 'imputed': 98}},
 {'RAVLT_forgetting': {'total_missing': 137, 'imputed': 107}},
 {'RAVLT_perc_forgetting': {'total_missing': 203, 'imputed': 150}},
 {'FAQ': {'total_missing': 34, 'imputed': 27}},
 {'MOCA': {'total_missing': 2687, 'imputed': 825}},
 {'LDELTOTAL': {'total_missing': 673, 'imputed

In [ ]:
#old
imputer_stat

[{'Ventricles': {'total_missing': 1343, 'imputed': 1024}},
 {'Hippocampus': {'total_missing': 1782, 'imputed': 1145}},
 {'WholeBrain': {'total_missing': 1202, 'imputed': 960}},
 {'Entorhinal': {'total_missing': 1989, 'imputed': 1144}},
 {'Fusiform': {'total_missing': 1989, 'imputed': 1144}},
 {'MidTemp': {'total_missing': 1989, 'imputed': 1144}},
 {'ICV': {'total_missing': 1061, 'imputed': 891}},
 {'FDG': {'total_missing': 3198, 'imputed': 495}},
 {'AV45': {'total_missing': 4034, 'imputed': 795}},
 {'CDRSB': {'total_missing': 46, 'imputed': 40}},
 {'MMSE': {'total_missing': 30, 'imputed': 26}},
 {'RAVLT_immediate': {'total_missing': 120, 'imputed': 98}},
 {'RAVLT_learning': {'total_missing': 120, 'imputed': 98}},
 {'RAVLT_forgetting': {'total_missing': 137, 'imputed': 107}},
 {'RAVLT_perc_forgetting': {'total_missing': 203, 'imputed': 150}},
 {'FAQ': {'total_missing': 34, 'imputed': 27}},
 {'MOCA': {'total_missing': 2693, 'imputed': 830}},
 {'LDELTOTAL': {'total_missing': 675, 'imputed

In [4]:
df_dataset.to_csv('dataset_imputed_final.csv', index=False)